In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Trade Path and MAE Analysis\n",
    "\n",
    "This notebook samples random trades from the backtest and visualizes their entry-to-exit PnL paths, highlighting maximum adverse excursion (MAE)."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "\n",
    "# Load trades data\n",
    "trades = pd.read_csv('../../../../results/data/trades_data.csv', parse_dates=['entry_date', 'exit_date'])\n",
    "print(trades.head())"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load price data\n",
    "prices = pd.read_csv('../../../../data/prices.csv', index_col=0, parse_dates=True)\n",
    "print(prices.head())"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def plot_trade_path(trade, prices):\n",
    "    t1, t2 = trade['pair1'], trade['pair2']\n",
    "    beta = trade.get('beta', 1.0)\n",
    "    entry_date = pd.to_datetime(trade['entry_date'])\n",
    "    exit_date = pd.to_datetime(trade['exit_date'])\n",
    "    # Get price data for the trade period\n",
    "    price_slice = prices.loc[entry_date:exit_date, [t1, t2]].copy()\n",
    "    spread = price_slice[t1] - beta * price_slice[t2]\n",
    "    pnl_path = spread - spread.iloc[0]\n",
    "    adverse = pnl_path.cummin()\n",
    "    mae = adverse.min()\n",
    "    mae_idx = adverse.idxmin()\n",
    "    plt.figure(figsize=(10, 4))\n",
    "    plt.plot(pnl_path, label='PnL Path (Spread)')\n",
    "    plt.scatter([pnl_path.index[0]], [0], color='green', label='Entry')\n",
    "    plt.scatter([pnl_path.index[-1]], [pnl_path.iloc[-1]], color='blue', label='Exit')\n",
    "    plt.scatter([mae_idx], [mae], color='red', label='MAE')\n",
    "    plt.axhline(0, color='gray', linestyle='--')\n",
    "    plt.title(f\"Trade: {t1} - {beta:.2f}*{t2} | Entry: {entry_date.date()} | Exit: {exit_date.date()}\")\n",
    "    plt.xlabel('Date')\n",
    "    plt.ylabel('Spread PnL (relative to entry)')\n",
    "    plt.legend()\n",
    "    plt.tight_layout()\n",
    "    plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Sample N random trades and plot\n",
    "N = 5  # Number of trades to plot\n",
    "sampled_trades = trades.sample(N, random_state=42)\n",
    "for idx, trade in sampled_trades.iterrows():\n",
    "    plot_trade_path(trade, prices)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# (Optional) Compute and plot MAE distribution for all trades\n",
    "def compute_mae(trade, prices):\n",
    "    t1, t2 = trade['pair1'], trade['pair2']\n",
    "    beta = trade.get('beta', 1.0)\n",
    "    entry_date = pd.to_datetime(trade['entry_date'])\n",
    "    exit_date = pd.to_datetime(trade['exit_date'])\n",
    "    price_slice = prices.loc[entry_date:exit_date, [t1, t2]].copy()\n",
    "    spread = price_slice[t1] - beta * price_slice[t2]\n",
    "    pnl_path = spread - spread.iloc[0]\n",
    "    mae = pnl_path.cummin().min()\n",
    "    return mae\n",
    "\n",
    "trades['mae'] = trades.apply(lambda row: compute_mae(row, prices), axis=1)\n",
    "print(trades['mae'].describe())\n",
    "plt.hist(trades['mae'], bins=30)\n",
    "plt.title('Distribution of Maximum Adverse Excursion (MAE)')\n",
    "plt.xlabel('MAE')\n",
    "plt.ylabel('Number of Trades')\n",
    "plt.show()"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.8"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 2
}
